# Suite SVC — The SystemVerilog Constraints dialect

The constraints and distributions of constraint blocks (IEEE 1800 18.5), a dialect of SystemVerilog's family,
`Expressions.SystemVerilog.Constraints`, as mbse-programs has them: its union of its kinds, listing every branch of
SystemVerilog's `Expressions`, and its holder; each kind's shape, its checks and what it binds (a `foreach` its loop
variables, refining `Let`); trees that mix its kinds with expressions, held by `randomize() with` and by an assertion's
operand; invalid syntax; the editions of IEEE 1800 it covers; printing, as mbse-programs' printer writes its kinds; and
evaluation, which refuses each kind by name. The family's `Expressions` and `Types` are suite DIA's.

In [ ]:
import functools

from mbse.Expressions import Framework
from mbse.Expressions.IEEE1164 import Expressions as IEEE1164
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Unicode import Expressions as Unicode
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3 import Expressions as P
from mbse.Expressions.SystemVerilog import Assertions as SVAssertions, Constraints as SVConstraints, \
    Expressions as SystemVerilog, Patterns as SVPatterns, Preprocessor as SVPreprocessor, Types as SVTypes
from mbse.Expressions.SystemVerilog.Assertions import Expressions as SVA
from mbse.Expressions.SystemVerilog.Constraints import Expressions as SVC, Text as SVCText
from mbse.Expressions.SystemVerilog.Expressions import Simplifiers as SVV, Expressions as SV, Text as SVText
from mbse.Expressions.SystemVerilog.Patterns import Expressions as SVP
from mbse.Expressions.SystemVerilog.Preprocessor import Expressions as SVPP
from mbse.Expressions.SystemVerilog.Types import Expressions as SVT
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises
from mbse.Expressions.Framework.Simplifiers import NotEvaluated

store = Builders.Store()  # SystemVerilog's family, after the framework and the standard dialects; Python3 for a foreign term
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, Patterns.install,
                Python3.install, SystemVerilog.install, SVTypes.install, SVPatterns.install, SVAssertions.install,
                SVConstraints.install, SVPreprocessor.install, PythonTypes.install, PythonPatterns.install]:  # Python's
    store.transaction(install)  # new dialects after its family's dialect
for define in [SystemVerilog.define, SVConstraints.define]:  # SystemVerilog's types, which its validation of a cast reads,
    store.transaction(define)  # and the editions the Constraints dialect covers
FAMILY = [SV.DIALECT, SVT.DIALECT, SVP.DIALECT, SVA.DIALECT, SVC.DIALECT, SVPP.DIALECT]


def held(spec, dialect=SVC.DIALECT):
    """The holder of the term `spec` writes: the dialect's own `Expression`, written in a transaction."""
    holder = functools.reduce(getattr, dialect.path.split("."), store.Expressions).Expression
    with store.transaction():
        return holder(lambda e: e.root(dialect.spec(spec)))


def term(spec, dialect=SVC.DIALECT):
    """The term `spec` writes, the root of its dialect's holder."""
    return F.term(held(spec, dialect))


def validate(spec, bound=(), dialect=SVC.DIALECT):
    return dialect.validate(term(spec, dialect), bound)


def text(spec, dialect=SVC.DIALECT):
    """The term `spec` writes, as SystemVerilog source, printed from its dialect's holder."""
    return SVCText.ToText(held(spec, dialect))


def schemas(dialect):
    """A dialect's `Term`'s branches' schemas, by tag."""
    return {b.name: b.schema for b in store.registered(dialect.union).branches}


union = lambda dialect: list(schemas(dialect))  # noqa: E731: its Term's branches
spelled = lambda spelling: lambda i: i.spelling(spelling)  # noqa: E731: an identifier, of its spelling
name = lambda spelling: lambda u: u.nameexpression(lambda n: n.name(lambda i: i.identifier(spelled(spelling))))  # noqa: E731
holds = lambda expression: lambda u: u.expressionconstraint(lambda c: c.expression(expression))  # noqa: E731: `expression;`
block = lambda *items: lambda u: u.constraintblock(lambda b: b.items(list(items)))  # noqa: E731: `{ items }`
x, y, a, i, j = name("x"), name("y"), name("a"), name("i"), name("j")

<a id="SVC-01"></a>
## SVC-01 · Its union lists Expressions' branches and its own kinds, and Assertions' a distribution; its holder is validated, linked by `HasRoot` and round-tripped

In [ ]:
# Its Term lists Expressions' branches, which that install adds, then its own kinds, then the Preprocessor's macro usage;
# Assertions' lists a distribution too, as a sequence's expression may be one (16.7), each schema one in both unions.
assert SVC.TAGS == ("constraintblock", "expressionconstraint", "implicationconstraint", "conditionalconstraint",
                    "foreachconstraint", "solvebeforeconstraint", "disablesoftconstraint", "uniqueconstraint", "distexpression")
own = len(SV.TAGS) + 2 + len(SVC.TAGS)
assert union(SVC.DIALECT)[:own + 1] == [*SV.TAGS, "error", "missing", *SVC.TAGS, "macrousage"]
assert union(SVA.DIALECT)[-2:] == ["distexpression", "macrousage"] and "distexpression" not in union(SV.DIALECT)
assert schemas(SVA.DIALECT)["distexpression"] is schemas(SVC.DIALECT)["distexpression"]
assert all(schemas(SVC.DIALECT)[tag] is schemas(SV.DIALECT)[tag] for tag in SV.TAGS)
assert list(SVC.DIALECT.kinds()) == [*SVC.TAGS, "error", "missing"] and SVC.HELPERS == ("DistItem",)
DistItem = store.registered("Expressions.SystemVerilog.Constraints.DistItem")
assert not DistItem.ref and "distitem" not in union(SVC.DIALECT)  # a helper: a value object, no kind
# Its dialect: named by its family, a branch of the framework's Term and of its kind union of dialects, a singleton.
assert SVC.DIALECT.name() == "SystemVerilog Constraints" and SVC.DIALECT.tag == "systemverilog_constraints"
assert {b.name: b.schema for b in store.registered("Expressions.Framework.Term").branches}["systemverilog_constraints"] is \
    store.registered(SVC.DIALECT.union)
assert {b.name: b.schema for b in store.registered("Expressions.Framework.Dialect").branches}["systemverilog_constraints"] is \
    store.registered("Expressions.SystemVerilog.Constraints.Dialect")
assert store.singleton("Expressions.SystemVerilog.Constraints") is store.Instances.Expressions.SystemVerilog.Constraints
assert SVC.DIALECT.reads() == tuple(d for d in FAMILY if d is not SVC.DIALECT)
# Its holder: a reference object refining the framework's Expression, its root narrowed to its Term. A constraint block
# held is validated, linked by HasRoot and round-tripped through JSON and YAML.
Holder = store.registered("Expressions.SystemVerilog.Constraints.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(SVC.DIALECT.union)
constraints = held(block(holds(SV.binary(">", x, 0)), lambda u: u.distexpression(lambda d: d.value(y).items([
    lambda e: e.value(SV.literal(0)).operator(":=").weight(SV.literal(3))]))))
with store.transaction():
    linked = store.Schemas.OfExpression(lambda s: s.name("svc.Randomized").expression(
        lambda e: e.insert(lambda h: h.expression(constraints))))
assert [F.same(entry.expression, constraints) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
assert Validators.Validate(store)(Holder, constraints) == [] and SVC.DIALECT.validate(constraints, {"x", "y"}) == []
assert F.dialect_of(F.term(constraints)) == SVC.DIALECT.union
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, constraints)).root, constraints.root)
assert F.same(YAML.FromYAML(store)(Holder, YAML.ToYAML(store)(Holder, constraints)).root, constraints.root)
Term = store.registered(SVC.DIALECT.union)  # its Term alone, as any union value
assert F.same(JSON.FromJSON(store)(Term, JSON.ToJSON(store)(Term, F.term(constraints))), F.term(constraints))
# Its root holds its Term's kinds alone: a Types term is refused by the schema, and another family's term by its
# validation, as it refuses its own to the other.
with raises(Exception, match="'structtype' is not a member of a union value"):
    held(SVC.DIALECT.make(F.Form("structtype", {"keyword": "struct"})))
python = F.term(held(1, P.DIALECT))
assert SVC.DIALECT.validate(python) == ["a Python3 literal is not a SystemVerilog Constraints term"]
assert P.DIALECT.validate(F.term(constraints)) == ["a SystemVerilog Constraints constraintblock is not a Python3 term"]

<a id="SVC-02"></a>
## SVC-02 · A constraint block holds constraints, and an expression constraint a distribution or an expression, `soft` or not

In [ ]:
# A constraint block holds its items, constraints, as its Term (mbse-programs' `Constraint | Directive | Comment`); an
# expression constraint its expression, a distribution or an expression (`expression_or_dist`), and `soft`, a bool.
kinds = schemas(SVC.DIALECT)
Term = store.registered(SVC.DIALECT.union)
assert list(kinds["constraintblock"].properties) == ["items"] and kinds["constraintblock"].properties["items"].schema.element is Term
assert list(kinds["expressionconstraint"].properties) == ["soft", "expression"]
assert kinds["expressionconstraint"].properties["expression"].schema is Term
assert F._parts(kinds["constraintblock"]) == ((), ("items",), ()) and F._parts(kinds["expressionconstraint"]) == (("expression",), (), ("soft",))
soft = lambda u: u.expressionconstraint(lambda c: c.soft(True).expression(SV.binary(">", x, 0)))  # noqa: E731
for spec, problems in [
    (block(), []), (block(soft, block(holds(y))), []),
    (holds(lambda u: u.distexpression(lambda d: d.value(x).items([lambda e: e.value(SV.literal(1))]))), []),
    (lambda u: u.expressionconstraint(lambda c: c.soft(True)), ["an expressionconstraint needs an expression"]),
    (lambda u: u.expressionconstraint(lambda c: c.soft("yes").expression(x)), ["an expressionconstraint's soft must be a bool, got str"]),
    (block(holds(name("z"))), ["argument 0: expression: nameexpression 'z' is not bound"]),  # its items' terms, in scope
]:
    assert validate(spec, {"x", "y"}) == problems, (problems, validate(spec, {"x", "y"}))
assert SY.free(term(block(soft, holds(y))), SVC.DIALECT) == {"x", "y"} and F.name_of(term(soft)) is None
assert SVC.DIALECT.infer(term(soft), {"x": D.Anything}) is D.Anything  # a constraint's domain is not known statically

<a id="SVC-03"></a>
## SVC-03 · An implication's and a conditional's condition is an expression, their bodies constraints, an alternative optional

In [ ]:
# An implication's and a conditional's condition is an expression, Expressions' Term; its body, consequence and
# alternative are constraints, its own Term; a conditional's alternative may be absent.
kinds = schemas(SVC.DIALECT)
ETerm, Term = store.registered(SV.DIALECT.union), store.registered(SVC.DIALECT.union)
assert list(kinds["implicationconstraint"].properties) == ["condition", "body"]
assert list(kinds["conditionalconstraint"].properties) == ["condition", "consequence", "alternative"]
assert kinds["implicationconstraint"].properties["condition"].schema is ETerm and kinds["implicationconstraint"].properties["body"].schema is Term
assert [kinds["conditionalconstraint"].properties[p].schema for p in ("condition", "consequence", "alternative")] == [ETerm, Term, Term]
implies = lambda condition, body: lambda u: u.implicationconstraint(lambda c: c.condition(condition).body(body))  # noqa: E731
when = lambda condition, consequence: lambda u: u.conditionalconstraint(lambda c: c.condition(condition).consequence(consequence))  # noqa: E731
for spec, problems in [
    (implies(x, holds(y)), []), (when(x, holds(y)), []),
    (lambda u: u.conditionalconstraint(lambda c: c.condition(x).consequence(holds(y)).alternative(when(y, block()))), []),
    (lambda u: u.implicationconstraint(lambda c: c.body(holds(y))), ["an implicationconstraint needs a condition"]),
    (lambda u: u.implicationconstraint(lambda c: c.condition(x)), ["an implicationconstraint needs a body"]),
    (lambda u: u.conditionalconstraint(lambda c: c.condition(x)), ["a conditionalconstraint needs a consequence"]),
    (implies(name("z"), holds(y)), ["condition: nameexpression 'z' is not bound"]),
    (when(x, holds(name("z"))), ["consequence: expression: nameexpression 'z' is not bound"]),
]:
    assert validate(spec, {"x", "y"}) == problems, (problems, validate(spec, {"x", "y"}))
# An expression subterm is of Expressions' terms alone: a constraint there is refused by the schema.
with raises(Exception, match="'constraintblock' is not a member of a union value"):
    held(SVC.DIALECT.make(F.Form("implicationconstraint", {}, (block(), holds(y)))))

<a id="SVC-04"></a>
## SVC-04 · A `foreach` refines `Let`: its array its value, its body its body, binding its loop variables within its body alone

In [ ]:
# A loop binds its variables within its body (18.5.8.1): it refines the framework's Let (Q8, "(a)"), its array Let's value,
# an Expressions term, its body Let's body, its own Term; Let's name stays unset, its kind's scopes saying what it binds.
# Its variables are each an identifier, or an empty argument for a dimension it skips (mbse-programs'
# `Identifier | EmptyArgument`), its union LoopVariable.
Foreach = schemas(SVC.DIALECT)["foreachconstraint"]
assert Foreach.refines is store.registered("Expressions.Framework.Let") and F.is_binder(Foreach)
assert list(Foreach.properties) == ["name", "value", "body", "variables"]  # Let's, then its own
assert F._parts(Foreach) == (("value", "body"), (), ("name", "variables"))
assert Foreach.properties["value"].schema is store.registered(SV.DIALECT.union)
assert Foreach.properties["body"].schema is store.registered(SVC.DIALECT.union)
LoopVariable = store.registered("Expressions.SystemVerilog.Constraints.LoopVariable")
assert Foreach.properties["variables"].schema.element is LoopVariable
assert [b.name for b in LoopVariable.branches] == ["identifier", "emptyargument"]
assert LoopVariable.branches[0].schema is store.registered("Expressions.SystemVerilog.Expressions.Identifier")
assert SVC.DIALECT.kinds()["foreachconstraint"].role == F.BINDING
loop = lambda array, variables, body: lambda u: u.foreachconstraint(lambda f: f.value(array).variables(variables).body(body))  # noqa: E731
at = lambda value, index: lambda u: u.indexexpression(lambda e: e.value(value).index(index))  # noqa: E731
skipped = lambda v: v.emptyargument(lambda e: e)  # noqa: E731
each = term(loop(a, [lambda v: v.identifier(spelled("i")), skipped, lambda v: v.identifier(spelled("j"))],
                 holds(SV.binary("<", at(a, i), j))))
assert SY.free(each, SVC.DIALECT) == {"a"} and SVC.DIALECT.validate(each, {"a"}) == [] and F.name_of(each) is None
assert SVC.loop_variables(F.kind_of(each)[1]) == {"i", "j"}
assert SVC.DIALECT.kinds()["foreachconstraint"].scopes(F.kind_of(each)[1]) == [frozenset(), frozenset({"i", "j"})]
assert SVC.DIALECT.infer(each, {"a": D.Anything}) is D.Anything
for spec, problems in [
    (loop(at(a, i), [lambda v: v.identifier(spelled("i"))], holds(i)), ["value: index: nameexpression 'i' is not bound"]),  # not in its array
    (loop(a, [lambda v: v.identifier(spelled("1i"))], block()), ["'1i' is not an identifier"]),
    (loop(a, [lambda v: v.identifier(lambda s: s)], holds(i)), ["body: expression: nameexpression 'i' is not bound"]),  # no spelling: binds none
    (loop(a, [skipped], holds(i)), ["body: expression: nameexpression 'i' is not bound"]),
    (lambda u: u.foreachconstraint(lambda f: f.body(block())), ["a foreachconstraint needs a value"]),
    (lambda u: u.foreachconstraint(lambda f: f.value(a)), ["a foreachconstraint needs a body"]),
]:
    assert validate(spec, {"a"}) == problems, (problems, validate(spec, {"a"}))

<a id="SVC-05"></a>
## SVC-05 · `solve ... before` orders two lists of variables, `disable soft` names its target, and `unique` its set of expressions and ranges

In [ ]:
# `solve ... before` holds two lists of expressions, the variables solved first and those after (18.5.10), two
# containers of terms, each read apart; `disable soft` names its target, an expression; `unique` its set, expressions
# and ranges (IEEE 1800-2012, 18.5.5).
kinds = schemas(SVC.DIALECT)
ETerm = store.registered(SV.DIALECT.union)
assert list(kinds["solvebeforeconstraint"].properties) == ["solve", "before"]
assert all(kinds["solvebeforeconstraint"].properties[p].schema.element is ETerm for p in ("solve", "before"))
assert F._parts(kinds["solvebeforeconstraint"]) == ((), ("solve", "before"), ())
order = lambda solve, before: lambda u: u.solvebeforeconstraint(lambda s: s.solve(solve).before(before))  # noqa: E731
node = F.node(*F.kind_of(term(order([x, y], [a]))))
assert [F.form(t).attributes["name"].attributes["spelling"] for t in node.lists["solve"]] == ["x", "y"] and len(node.lists["before"]) == 1
assert F.form(term(order([x, y], [a]))).attributes == {"solve": F.Part("solve", {}, 2), "before": F.Part("before", {}, 1)}
assert F.same(term(SVC.DIALECT.make(F.form(term(order([x, y], [a]))))), term(order([x, y], [a])))
assert list(kinds["disablesoftconstraint"].properties) == ["target"] and kinds["disablesoftconstraint"].properties["target"].schema is ETerm
assert list(kinds["uniqueconstraint"].properties) == ["set"] and kinds["uniqueconstraint"].properties["set"].schema.element is ETerm
assert F._parts(kinds["uniqueconstraint"]) == ((), ("set",), ())
span = lambda u: u.valuerange(lambda r: r.left(SV.literal(1)).right(SV.literal(3)))  # noqa: E731: [1:3]
for spec, problems in [
    (order([x, y], [a]), []), (order([], []), []),
    (order([x], [name("z")]), ["before 0: nameexpression 'z' is not bound"]),
    (lambda u: u.disablesoftconstraint(lambda d: d.target(x)), []),
    (lambda u: u.uniqueconstraint(lambda q: q.set([x, a, span])), []), (lambda u: u.uniqueconstraint(lambda q: q.set([])), []),
    (lambda u: u.disablesoftconstraint(lambda d: d), ["a disablesoftconstraint needs a target"]),
    (lambda u: u.uniqueconstraint(lambda q: q.set([x, name("z")])), ["argument 1: nameexpression 'z' is not bound"]),
    (lambda u: u.uniqueconstraint(lambda q: q.set([lambda v: v.valuerange(lambda r: r.left(x).operator("+-").right(y))])),
     ["argument 0: ValueRange.operator cannot be '+-'"]),
]:
    assert validate(spec, {"x", "y", "a"}) == problems, (problems, validate(spec, {"x", "y", "a"}))

<a id="SVC-06"></a>
## SVC-06 · A distribution's items, each a value or `default` with an operator and a weight, checked as mbse-programs checks them; Assertions' subterms may be one

In [ ]:
# A distribution (18.5.4): its value an expression, its items helpers, each its value (an expression or a range), or none
# for `default`, and its operator and weight, both or neither, as mbse-programs checks them.
Dist = schemas(SVC.DIALECT)["distexpression"]
ETerm = store.registered(SV.DIALECT.union)
assert list(Dist.properties) == ["value", "items"] and Dist.properties["value"].schema is ETerm
assert Dist.properties["items"].schema.element is DistItem and F._parts(Dist) == (("value",), (), ("items",))
assert list(DistItem.properties) == ["value", "operator", "weight"]
assert DistItem.properties["value"].schema is DistItem.properties["weight"].schema is ETerm and SVC.OPERATORS == (":=", ":/")
dist = lambda *items: lambda u: u.distexpression(lambda d: d.value(x).items(list(items)))  # noqa: E731
item = lambda value=None, operator=None, weight=None: lambda e: functools.reduce(  # noqa: E731: an item, of what it has
    lambda b, part: getattr(b, part[0])(part[1]), [p for p in (("value", value), ("operator", operator), ("weight", weight))
                                                     if p[1] is not None], e)
for spec, problems in [
    (dist(item(SV.literal(0), ":=", SV.literal(40)), item(span, ":/", SV.literal(60)), item(y), item(None, ":/", SV.literal(1))), []),
    (dist(item(SV.literal(0), "=", SV.literal(1))), ["DistItem.operator cannot be '='"]),
    (dist(item(SV.literal(0), ":=")), ["a DistItem has both an operator and a weight, or neither"]),
    (dist(item(SV.literal(0), None, SV.literal(2))), ["a DistItem has both an operator and a weight, or neither"]),
    (dist(item()), ["a default DistItem weighs its values with :/"]),
    (dist(item(None, ":=", SV.literal(1))), ["a default DistItem weighs its values with :/"]),
    (dist(item(SV.literal(0), ":=", name("w"))), ["items 0 weight: nameexpression 'w' is not bound"]),
    (lambda u: u.distexpression(lambda d: d.items([item(y)])), ["a distexpression needs a value"]),
]:
    assert validate(spec, {"x", "y"}) == problems, (problems, validate(spec, {"x", "y"}))
# Assertions' Term lists it: a sequence's expression, a call's argument (`Property | Expression`), may be a distribution.
weighted = dist(item(SV.literal(1), ":=", SV.literal(2)))
assert validate(weighted, {"x"}, SVA.DIALECT) == [] and F.dialect_of(term(weighted, SVA.DIALECT)) == SVA.DIALECT.union
called = term(SV.call("f", weighted), SV.DIALECT)
assert F.dialect_of(F.form(called).arguments[1]) == SVA.DIALECT.union and SV.DIALECT.validate(called, {"f", "x"}) == []
assert SVA.DIALECT.kind(F.form(called).arguments[1])[2] is SVC.DIALECT.kinds()["distexpression"]  # read as its dialect reads it
with raises(Exception, match="'distexpression' is not a member of a union value"):  # Expressions' Term lists it not
    held(SV.DIALECT.make(F.Form("distexpression", {}, (x,))), SV.DIALECT)

<a id="SVC-07"></a>
## SVC-07 · A tree mixing constraints and expressions, held by `randomize() with` or by its own holder, validated and printed from either, each dialect of the family reading it

In [ ]:
# A tree mixing its kinds with expressions: `randomize() with` holds constraints, its own Term, which hold expressions,
# Expressions' Term, and a distribution; validated, its free names found and printed from Expressions' holder, through
# the property that holds its Term as a subterm, and from its own holder, each dialect reading the others' terms.
randomize = lambda *items: lambda u: u.randomizewithexpression(lambda r: r.call(SV.method(x, "randomize")).items(list(items)))  # noqa: E731
loop = lambda u: u.foreachconstraint(lambda f: f.value(a).variables([lambda v: v.identifier(spelled("i"))]).body(  # noqa: E731
    holds(lambda d: d.distexpression(lambda e: e.value(lambda q: q.indexexpression(lambda s: s.value(a).index(i))).items([
        lambda t: t.value(SV.literal(0)).operator(":=").weight(name("w"))])))))
mixed = term(randomize(loop, lambda u: u.implicationconstraint(lambda c: c.condition(SV.inside(y, SV.span(1, 3))).body(
    holds(SV.binary("==", name("z"), 0))))), SV.DIALECT)
assert SV.DIALECT.validate(mixed, {"x", "y", "a", "w", "z"}) == [] and SY.free(mixed, SV.DIALECT) == {"x", "y", "a", "w", "z"}
items = F.form(mixed).arguments[1:]
assert [F.dialect_of(t) for t in items] == [SVC.DIALECT.union] * 2 and SV.DIALECT.kind(items[0])[2] is SVC.DIALECT.kinds()["foreachconstraint"]
assert SV.DIALECT.validate(mixed, {"x", "y", "a", "z"}) == [
    "argument 0: body: expression: items 0 weight: nameexpression 'w' is not bound"]
assert SVText.ToText(mixed) == "x.randomize() with { foreach (a[i]) a[i] dist {0 := w}; y inside {[1:3]} -> z == 0; }"
assert SVC.DIALECT.infer(items[0], {"a": D.Anything, "w": D.Anything}) is D.Anything
# From its own holder: a block whose items hold expressions, and an expression whose argument is its own term again
nested = held(block(holds(lambda u: u.callexpression(lambda c: c.callee(name("f")).arguments([lambda d: d.distexpression(
    lambda e: e.value(x).items([lambda t: t.value(SV.literal(1))]))]))), randomize(holds(y))))
assert SVC.DIALECT.validate(nested, {"f", "x", "y"}) == [] and SY.free(F.term(nested), SVC.DIALECT) == {"f", "x", "y"}
assert SVCText.ToText(nested) == "{ f(x dist {1}); x.randomize() with { y; }; }"
assert all(d.validate(F.term(nested), {"f", "x", "y"}) == [] for d in FAMILY)  # each dialect of the family reads it alike
assert SVC.DIALECT.validate(F.term(nested), {"x", "y"}) == ["argument 0: expression: callee: nameexpression 'f' is not bound"]

<a id="SVC-08"></a>
## SVC-08 · Invalid syntax stands wherever a constraint or an expression does: reported, printed as written, left in the residual by simplification, named

In [ ]:
# Invalid syntax, `error` and `missing`, stands wherever its Term or Expressions' does: reported, printed as written,
# refused by evaluation by name.
error = lambda text: lambda u: u.error(lambda e: e.text(text))  # noqa: E731
missing = lambda u: u.missing(lambda m: m.kind("Constraint"))  # noqa: E731
for spec, problems, source in [
    (error("x >"), ["invalid syntax 'x >'"], "x >"),
    (block(holds(x), missing), ["argument 1: missing Constraint"], "{ x;  }"),
    (lambda u: u.implicationconstraint(lambda c: c.condition(error("a &")).body(holds(x))), ["condition: invalid syntax 'a &'"], "a & -> x;"),
    (lambda u: u.foreachconstraint(lambda f: f.value(a).body(error("??"))), ["body: invalid syntax '??'"], "foreach (a[]) ??"),
    (lambda u: u.distexpression(lambda d: d.value(x).items([lambda t: t.value(error("1:")).operator(":=").weight(SV.literal(1))])),
     ["items 0 value: invalid syntax '1:'"], "x dist {1: := 1}"),
]:
    assert validate(spec, {"x", "a"}) == problems, (problems, validate(spec, {"x", "a"}))
    assert text(spec) == source, (source, text(spec))
with raises(NotEvaluated, match="SystemVerilog Constraints error is not simplified"):  # invalid syntax stays, named
    SVV.OfAny(term(error("x >")), {}).evaluate()
with raises(ValueError, match="SystemVerilog Constraints error has no text to print"):
    text(lambda u: u.error(lambda e: e))

<a id="SVC-09"></a>
## SVC-09 · The editions it covers are data: IEEE Std 1800-2005 to 1800-2023, each with its own kinds from the edition it arrived in

In [ ]:
# Its editions are data (CAP-268): IEEE Std 1800-2005 to 1800-2023, owned by designation, each with its own kinds it has,
# from mbse-programs' SINCE: `disable soft` and `unique` from IEEE 1800-2012, the others from 1800-2005; its define runs
# again writing nothing, and binds each edition by its common name spelled as an identifier.
singleton = store.Instances.Expressions.SystemVerilog.Constraints
LABELS = ("SystemVerilog-2005", "SystemVerilog-2009", "SystemVerilog-2012", "SystemVerilog-2017", "SystemVerilog-2023")
assert tuple(FS.standards(singleton)) == tuple(f"IEEE Std 1800-{year}" for year in (2005, 2009, 2012, 2017, 2023))
assert SVC.SINCE == {"disablesoftconstraint": 5, "uniqueconstraint": 5} and SV.EDITIONS[5][0] == "IEEE Std 1800-2012"
assert FS.tags(FS.find(singleton, "SystemVerilog-2012")) == ("IEEE Std 1800-2012", "SystemVerilog-2012")
assert FS.properties(FS.find(singleton, "SystemVerilog-2005")) == {"bodies": ("org.ieee",), "number": "1800",
                                                                   "revision": ("year", 2005), "names": ("SystemVerilog-2005",)}
assert FS.find(singleton, "Verilog-2005") is None  # IEEE 1364's have no constraints
editions = {tag: tuple(FS.tags(s)[-1] for s in FS.of(singleton, schema).values()) for tag, schema in schemas(SVC.DIALECT).items()}
assert editions == {tag: LABELS[SVC.SINCE.get(tag, 3) - 3:] if tag in SVC.TAGS else () for tag in union(SVC.DIALECT)}
assert [s for s in FS.kinds(FS.find(singleton, "SystemVerilog-2009"))] == [
    schemas(SVC.DIALECT)[tag] for tag in SVC.TAGS if tag not in SVC.SINCE]
frame = store.transaction(SVConstraints.define)  # again: nothing new
assert frame[0]["SystemVerilog_2023"] is FS.find(singleton, "SystemVerilog-2023") and len(FS.standards(singleton)) == 5

<a id="SVC-10"></a>
## SVC-10 · Printing every kind as `randomize() with` writes it, on one line, with IEEE 1800's precedence

In [ ]:
# Printing ports mbse-programs' printer (`constraint_text`, `DistExpression`): a constraint on one line, as
# `randomize() with` writes it; an implication's condition parenthesized below `->`'s precedence, a distribution's value
# below a relational operator's; a distribution binds as `inside` does, and held as a constraint it is ended by `;`.
x_gt = SV.binary(">", x, 0)
cases = [
    (block(), "{}"), (block(holds(x_gt), holds(y)), "{ x > 0; y; }"), (block(block(holds(x))), "{ { x; } }"),
    (lambda u: u.expressionconstraint(lambda c: c.soft(True).expression(x_gt)), "soft x > 0;"),
    (lambda u: u.expressionconstraint(lambda c: c.soft(False).expression(x_gt)), "x > 0;"),
    (lambda u: u.implicationconstraint(lambda c: c.condition(SV.binary("||", x, y)).body(holds(y))), "x || y -> y;"),
    (lambda u: u.implicationconstraint(lambda c: c.condition(SV.binary("->", x, y)).body(block(holds(y)))), "(x -> y) -> { y; }"),
    (lambda u: u.implicationconstraint(lambda c: c.condition(SV.conditional(x, y, 0)).body(holds(y))), "x ? y : 0 -> y;"),
    (lambda u: u.conditionalconstraint(lambda c: c.condition(x).consequence(holds(y))), "if (x) y;"),
    (lambda u: u.conditionalconstraint(lambda c: c.condition(x).consequence(block(holds(y))).alternative(
        lambda v: v.conditionalconstraint(lambda d: d.condition(y).consequence(holds(x)).alternative(holds(a))))),
     "if (x) { y; } else if (y) x; else a;"),
    (lambda u: u.foreachconstraint(lambda f: f.value(SV.member(a, "q")).variables([lambda v: v.emptyargument(lambda e: e),
        lambda v: v.identifier(spelled("j"))]).body(holds(j))), "foreach (a.q[, j]) j;"),
    (lambda u: u.foreachconstraint(lambda f: f.value(a).body(block())), "foreach (a[]) {}"),
    (lambda u: u.solvebeforeconstraint(lambda s: s.solve([x, y]).before([SV.member(a, "q")])), "solve x, y before a.q;"),
    (lambda u: u.disablesoftconstraint(lambda d: d.target(SV.member(a, "q"))), "disable soft a.q;"),
    (lambda u: u.uniqueconstraint(lambda q: q.set([x, a, span])), "unique {x, a, [1:3]};"),
    (lambda u: u.uniqueconstraint(lambda q: q.set([])), "unique {};"),
    (dist(item(SV.literal(0), ":=", SV.literal(40)), item(span, ":/", SV.binary("+", x, 1)), item(y), item(None, ":/", SV.literal(1))),
     "x dist {0 := 40, [1:3] :/ x + 1, y, default :/ 1}"),
    (lambda u: u.distexpression(lambda d: d.value(SV.binary("+", x, y)).items([item(y)])), "x + y dist {y}"),
    (lambda u: u.distexpression(lambda d: d.value(SV.binary("<", x, y)).items([item(y)])), "x < y dist {y}"),
    (lambda u: u.distexpression(lambda d: d.value(SV.binary("==", x, y)).items([item(y)])), "(x == y) dist {y}"),
    (block(dist(item(y)), holds(dist(item(y)))), "{ x dist {y}; x dist {y}; }"),  # held as a constraint, or by one
]
for spec, source in cases:
    assert text(spec) == source, (source, text(spec))
# A distribution's precedence is a relational operator's, where an Assertions' operand holds it (an expression's
# operands, Expressions' terms, hold none).
weighted = SVText.printer().node(term(dist(item(y))))
assert SVText.printer().precedence(weighted, "x dist {y}") == SVText.RELATIONAL
assert SVText.printer().operand(term(dist(item(y))), SVText.RELATIONAL + 1) == "(x dist {y})"
assert all(package.Text.ToText is SVText.ToText for package in (SVConstraints, SystemVerilog))  # one printer for the family

<a id="SVC-11"></a>
## SVC-11 · Simplification leaves each of its kinds in the residual, named

In [ ]:
# Its kinds are not evaluated (Q11): SystemVerilog's evaluator refuses each by name, from any holder, valid as it is.
samples = {
    "constraintblock": block(holds(SV.literal(1))),
    "expressionconstraint": holds(SV.literal(1)),
    "implicationconstraint": lambda u: u.implicationconstraint(lambda c: c.condition(SV.literal(1)).body(block())),
    "conditionalconstraint": lambda u: u.conditionalconstraint(lambda c: c.condition(SV.literal(1)).consequence(block())),
    "foreachconstraint": lambda u: u.foreachconstraint(lambda f: f.value(SV.literal(1)).body(block())),
    "solvebeforeconstraint": lambda u: u.solvebeforeconstraint(lambda s: s.solve([SV.literal(1)]).before([SV.literal(2)])),
    "disablesoftconstraint": lambda u: u.disablesoftconstraint(lambda d: d.target(SV.literal(1))),
    "uniqueconstraint": lambda u: u.uniqueconstraint(lambda q: q.set([SV.literal(1)])),
    "distexpression": lambda u: u.distexpression(lambda d: d.value(SV.literal(1)).items([lambda t: t.value(SV.literal(1))])),
}
assert list(samples) == list(SVC.TAGS)
for tag, spec in samples.items():
    assert validate(spec) == []
    with raises(NotEvaluated, match=f"SystemVerilog Constraints {tag} is not simplified"):
        SVV.OfAny(term(spec), {}).evaluate()
with raises(NotEvaluated, match="SystemVerilog Constraints distexpression is not simplified"):  # within a call
    SVV.OfAny(term(SV.call("f", samples["distexpression"]), SV.DIALECT), SVV.Scope({}, {"f": lambda value: value})).evaluate()